### Loading data

In [1]:
import pandas as pd

df_raw = pd.read_csv('../data/reestrtz31.12.2025.csv', delimiter=';')
df_raw.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5468\2489814255.py:3: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../data/reestrtz31.12.2025.csv', delimiter=';')


,PERSON,REG_ADDR_KOATUU,OPER_CODE,OPER_NAME,D_REG,DEP_CODE,DEP,BRAND,MODEL,VIN,MAKE_YEAR,COLOR,KIND,BODY,PURPOSE,FUEL,CAPACITY,OWN_WEIGHT,TOTAL_WEIGHT,N_REG_NEW
0,P,6.310138e+09,254,НАЛЕЖНИЙ КОРИСТУВАЧ. РЕЄСТРАЦІЯ,01.01.25,10000,OLD_ДДАІ МВС УКРАЇНИ,LEXUS,IS 250,JTHCK262882026114,2008,ЧЕРВОНИЙ,ЛЕГКОВИЙ,СЕДАН,ЗАГАЛЬНИЙ,БЕНЗИН,2499.0,2075,2575,АХ1312КК
1,P,3.222486e+09,254,НАЛЕЖНИЙ КОРИСТУВАЧ. РЕЄСТРАЦІЯ,01.01.25,10000,OLD_ДДАІ МВС УКРАЇНИ,MG,350,LSJA16E34EG012544,2014,ЧОРНИЙ,ЛЕГКОВИЙ,СЕДАН,ЗАГАЛЬНИЙ,БЕНЗИН,1498.0,1265,1640,АІ9214СІ
2,P,3.521486e+09,254,НАЛЕЖНИЙ КОРИСТУВАЧ. РЕЄСТРАЦІЯ,01.01.25,10000,OLD_ДДАІ МВС УКРАЇНИ,SKODA,SUPERB,TMBCR63U789012741,2007,ЧОРНИЙ,ЛЕГКОВИЙ,СЕДАН,ЗАГАЛЬНИЙ,ДИЗЕЛЬНЕ ПАЛИВО,1968.0,1593,2103,АХ1298ОА
3,P,8.000000e+09,254,НАЛЕЖНИЙ КОРИСТУВАЧ. РЕЄСТРАЦІЯ,01.01.25,10000,OLD_ДДАІ МВС УКРАЇНИ,TOYOTA,RAV-4 HYBRID,JTMW23FVX0D106165,2021,БІЛИЙ,ЛЕГКОВИЙ,УНІВЕРСАЛ,ЗАГАЛЬНИЙ,ЕЛЕКТРО АБО БЕНЗИН,2487.0,1755,2135,КА5010НС
4,P,5.120410e+09,254,НАЛЕЖНИЙ КОРИСТУВАЧ. РЕЄСТРАЦІЯ,01.01.25,10000,OLD_ДДАІ МВС УКРАЇНИ,RENAULT,KANGOO,VF1KW28B551509043,2014,БІЛИЙ,ЛЕГКОВИЙ,УНІВЕРСАЛ,ЗАГАЛЬНИЙ,ДИЗЕЛЬНЕ ПАЛИВО,1461.0,1395,1950,НН1103АС


### Cleaning

Dropping the columns we don't need

In [2]:
df = df_raw.drop(columns=[
    'OPER_CODE', 'DEP_CODE', 'DEP', 'VIN', 'CAPACITY', 'OWN_WEIGHT', 'TOTAL_WEIGHT'
])

Converting registration address from float to string and registration date from string to datetime

In [3]:
df['D_REG'] = pd.to_datetime(df['D_REG'], format='%d.%m.%y')

Keeping only passenger cars

In [4]:
df = df[df['KIND'] == 'ЛЕГКОВИЙ'].copy()
df = df.drop(columns=['KIND'])

Removing special/comercial body types

In [5]:
target_bodies = [
    'УНІВЕРСАЛ', 'СЕДАН', 'ХЕТЧБЕК', 'ПАСАЖИРСЬКИЙ', 'КУПЕ',
    'ВАНТАЖОПАСАЖИРСЬКИЙ', 'ПІКАП', 'КАБРІОЛЕТ', 'ФАЕТОН'
]

df = df[df['BODY'].isin(target_bodies)].copy()

Fixing exponential notation and convert KOATUU geographic codes to clean string format

In [6]:
df['REG_ADDR_KOATUU'] = df['REG_ADDR_KOATUU'].fillna(0).astype('int64').astype(str).str.zfill(10)

The Tableau dashboard will focus on car purchases in Ukraine in 2025, so we need to exclude all operations not related to buying.

In [7]:
target_operations = [
    'ПЕРЕРЕЄСТРАЦІЯ ТЗ НА НОВ. ВЛАСН. ПО ДОГОВОРУ УКЛАДЕНОМУ В ТСЦ',
    'ПЕРЕРЕЄСТРАЦІЯ НА НОВОГО ВЛАСНИКА ЗА ДКП ТА ДОГОВОРОМ КОМІСІЇ, УКЛАДЕНИМИ У СГ',
    'ПЕРЕРЕЄСТРАЦІЯ ТЗ ІЗ УКЛАДАННЯМ ДОГОВОРУ КУПІВЛІ-ПРОДАЖУ ЧЕРЕЗ ЕЛЕКТРОННІ СЕРВІСИ ТА ЗОВНІШНІ РЕСУРСИ',
    'ПЕРВИННА РЕЄСТРАЦIЯ Б/В ТЗ ПРИДБАНОГО В ТОРГОВЕЛЬНІЙ ОРГАНІЗАЦІЇ, ЯКИЙ ВВЕЗЕНО З-ЗА КОРДОНУ',
    'РЕЄСТРАЦIЯ Б/В ТЗ ПРИВЕЗЕНОГО З-ЗА КОРДОНУ ПО ВМД',
    'ПЕРВИННА РЕЄСТРАЦІЯ НОВОГО ТЗ ПРИДБАНОГО В ТОРГОВЕЛЬНІЙ ОРГАНІЗАЦІЇ, ЯКИЙ ВВЕЗЕНО З-ЗА КОРДОНУ',
    'РЕЄСТРАЦІЯ ТЗ ПРИВЕЗЕНОГО З-ЗА КОРДОНУ ПО ПОСВІДЧЕННЮ МИТНИЦІ',
    'ПЕРЕРЕЄСТРАЦІЯ ТЗ НА НОВОГО ВЛАСНИКА ПО ДОГОВОРУ КУПІВЛІ-ПРОДАЖУ УКЛАДЕНОМУ В ПРИСУТНОСТІ АДМІНІСТРАТОРА ЦНАП',
    'ПЕРЕРЕЄСТРАЦІЯ ТЗ НА НОВОГО ВЛАСНИКА ЗА ДОГОВОРОМ ФІНАНСОВОГО ЛІЗИНГУ ТА ДКП ПРЕДМЕТА ЛІЗИНГУ',
    'ПЕРЕРЕЄСТРАЦІЯ НА НОВОГО ВЛАСНИКА ЗА ДОГ. КУПІВЛІ-ПРОДАЖУ, УКЛАДЕНОГО У СГ НА ТЗ ЗАРЕЄСТРОВАНИЙ ЗА СГ',
    'РЕЄСТРАЦІЯ НОВОГО ТЗ, ПРИВЕЗЕНОГО З-ЗА КОРДОНУ ПО ВМД',
    'ПЕРЕРЕЄСТРАЦІЯ ТЗ НА НОВОГО ВЛАСНИКА ЗА АКТОМ ПРИЙОМУ-ПЕРЕДАЧІ',
    'ПЕРЕРЕЄСТРАЦІЯ НА НОВОГО ВЛАСНИКА ЗА ДОГ. КУП.-ПРОД.ТА ІНШ.(ОФОРМ.НОТАРІАЛЬНО )',
    'ПЕРВИННА РЕЄСТРАЦІЯ НОВОГО ТЗ ПРИДБАНОГО В ТОРГОВЕЛЬНІЙ ОРГАНІЗАЦІЇ, ЯКИЙ ВИГОТОВЛЕНО В УКРАЇНІ'    
]

df = df[df['OPER_NAME'].isin(target_operations)].copy()

### Saving

In [22]:
df = df.reset_index(drop=True)
df.to_csv('../data/cleaned_reestrtz31.12.2025.csv', index=False, sep=";")